# SmartFactorParams

The `SmartProjectionParams` class configures smart projection factors. It controls their linearization, triangulation, and degeneracy behavior.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/slam/doc/SmartFactorParams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import gtsam
import numpy as np

from gtsam.symbol_shorthand import X

## Configure SmartProjectionParams

We construct `SmartProjectionParams` with Jacobian-Q linearization and zero-information handling for degenerate triangulation. The setter methods then tune rank, epipolar-initialization, landmark-distance, and dynamic-outlier thresholds for the intended scene.

In [3]:
params = gtsam.SmartProjectionParams(
    gtsam.LinearizationMode.JACOBIAN_Q,
    gtsam.DegeneracyMode.ZERO_ON_DEGENERACY,
    False,
    False,
    1e-5,
)
params.setRankTolerance(1e-8)
params.setEnableEPI(False)
params.setLandmarkDistanceThreshold(1e4)
params.setDynamicOutlierRejectionThreshold(3.0)
params.print("configured smart-factor parameters")

linearizationMode: 2
   degeneracyMode: 1
rankTolerance = 1e-08


enableEPI = 0


landmarkDistanceThreshold = 10000


dynamicOutlierRejectionThreshold = 3


useLOST = 0


noise model


## Apply the parameters to a smart factor

Pass the configured object to a smart projection factor constructor. After adding two consistent camera observations, `linearizeToJacobian()` returns a `JacobianFactorQ`, confirming that the selected linearization mode took effect.

In [4]:
calibration = gtsam.Cal3_S2(500.0, 500.0, 0.0, 320.0, 240.0)
cameras = [
    gtsam.PinholeCameraCal3_S2(gtsam.Pose3(), calibration),
    gtsam.PinholeCameraCal3_S2(
        gtsam.Pose3(gtsam.Rot3(), np.array([1.0, 0.0, 0.0])),
        calibration,
    ),
]
landmark = np.array([0.0, 0.0, 5.0])
factor = gtsam.SmartProjectionFactorPinholeCameraCal3_S2(
    gtsam.noiseModel.Isotropic.Sigma(2, 1.0), params
)
values = gtsam.Values()
for index, camera in enumerate(cameras):
    factor.add(camera.project(landmark), X(index))
    values.insert(X(index), camera)

linear_factor = factor.linearizeToJacobian(values)
assert isinstance(linear_factor, gtsam.JacobianFactorQ112)
assert np.isclose(factor.totalReprojectionError(factor.cameras(values)), 0.0)
print("linear factor:", type(linear_factor).__name__)

linear factor: JacobianFactorQ112


## When to use it

Use `SmartProjectionParams` whenever a smart projection factor needs non-default triangulation, degeneracy, outlier, or linearization behavior. Start with the default Hessian mode unless the downstream solver specifically consumes Jacobian factors, and choose thresholds in the same scale as the scene and measurement noise.

[`SmartProjectionFactor`](SmartProjectionFactor.ipynb) shows the factor itself, while [`JacobianFactorQ`](JacobianFactorQ.ipynb) explains the linear factor returned by this example.

## Source

[`SmartFactorParams.h`](../SmartFactorParams.h)


## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.